In [ ]:
# Nacitanie dat — happiness_europe.txt
df <- read.table("Data/happiness_europe.txt", sep="\t", header=TRUE,
                 check.names=FALSE, stringsAsFactors=FALSE)

names(df) <- c("Country", "Region", "Happiness_Rank", "Happiness_Score",
               "Std_Error", "Economy", "Family", "Health",
               "Freedom", "Trust", "Generosity", "Dystopia_Residual")

# Konverzia numerickych stlpcov
num_cols <- c("Happiness_Rank","Happiness_Score","Std_Error","Economy",
              "Family","Health","Freedom","Trust","Generosity","Dystopia_Residual")
df[num_cols] <- lapply(df[num_cols], as.numeric)

cat("Rozmer:", nrow(df), "x", ncol(df), "\n")
str(df)

In [23]:
# MNS model: Happiness_Score ~ Family + Economy + Health + Freedom

Y <- df$Happiness_Score
X <- cbind(1, df$Family, df$Economy, df$Health, df$Freedom)
colnames(X) <- c("intercept", "Family", "Economy", "Health", "Freedom")

n <- nrow(X)
k <- ncol(X)

# odhad beta
betaHAT <- solve(t(X) %*% X) %*% t(X) %*% Y
rownames(betaHAT) <- colnames(X)
cat("Odhady beta:\n"); print(betaHAT)

# rezidua a odhad sigma^2
YHAT       <- X %*% betaHAT
epsHAT     <- Y - YHAT
s2         <- as.numeric(t(epsHAT) %*% epsHAT / (n - k))
cat("\nOdhad sigma^2 (s2):", s2, "\n")
cat("Odhad sigma  (s) :", sqrt(s2), "\n")

# R^2, RSS, ESS, TSS
RSS <- sum(epsHAT^2)
TSS <- sum((Y - mean(Y))^2)
ESS <- TSS - RSS
R2  <- ESS / TSS
cat("\nRSS:", RSS, " ESS:", ESS, " TSS:", TSS, "\n")
cat("R^2:", R2, "\n")

# overenie cez lm()
model <- lm(Happiness_Score ~ Family + Economy + Health + Freedom, data = df)
summary(model)

ERROR: Error in solve(t(X) %*% X) %*% t(X) %*% Y: requires numeric/complex matrix/vector arguments


In [ ]:
# Test kontrastu: H0: beta_Health = 2 * beta_Economy
# t.j. beta_Health - 2*beta_Economy = 0
# poradie: (intercept, Family, Economy, Health, Freedom)

a <- c(0, 0, -2, 1, 0)
r <- 0

X <- model.matrix(model)
T <- (t(a) %*% betaHAT - r) / sqrt(s2 * t(a) %*% solve(t(X)%*%X) %*% a)
krit <- qt(0.975, df = n-k)
p    <- 2 * (1 - pt(abs(T), df = n-k))

cat("T-statistika:    ", round(T, 4), "\n")
cat("Kritická hodnota:", round(krit, 4), "\n")
cat("p-hodnota:       ", round(p, 4), "\n\n")
if (p < 0.05) {
  cat("Záver: H0 ZAMIETAME (p < 0.05) — beta_Health != 2 * beta_Economy\n")
} else {
  cat("Záver: H0 NEZAMIETAME (p >= 0.05) — nedostatok dôkazov proti beta_Health = 2 * beta_Economy\n")
}

In [ ]:
# Test kontrastu: H0: beta_Family = beta_Economy + beta_Freedom
# t.j. beta_Family - beta_Economy - beta_Freedom = 0
# poradie: (intercept, Family, Economy, Health, Freedom)

a <- c(0, 1, -1, 0, -1)
r <- 0

X <- model.matrix(model)
T <- (t(a) %*% betaHAT - r) / sqrt(s2 * t(a) %*% solve(t(X)%*%X) %*% a)
krit <- qt(0.975, df = n-k)
p    <- 2 * (1 - pt(abs(T), df = n-k))

cat("T-statistika:    ", round(T, 4), "\n")
cat("Kritická hodnota:", round(krit, 4), "\n")
cat("p-hodnota:       ", round(p, 4), "\n\n")
if (p < 0.05) {
  cat("Záver: H0 ZAMIETAME (p < 0.05) — beta_Family != beta_Economy + beta_Freedom\n")
} else {
  cat("Záver: H0 NEZAMIETAME (p >= 0.05) — nedostatok dôkazov proti beta_Family = beta_Economy + beta_Freedom\n")
}